# Day 44 — Model deployment basics: save model & FastAPI
Objectives:
- Save/load models (joblib).
- Build a minimal FastAPI endpoint.
- Send a sample request.
Note: Running the server requires a terminal (see code comments).


In [ ]:
from pathlib import Path

import joblib
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

artifact_dir = Path('artifacts/day44')
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / 'model.joblib'
X,y = load_iris(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y,random_state=42)
clf = LogisticRegression(max_iter=1000).fit(Xtr,ytr)
print('test acc:', clf.score(Xte,yte))
joblib.dump(clf, model_path)


## Minimal FastAPI app (app.py)
Create a file `app.py` in this folder:
```python
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI()
model = joblib.load(model_path)

class IrisFeatures(BaseModel):
    features: list

@app.post('/predict')
def predict(data: IrisFeatures):
    X = np.array([data.features])
    pred = model.predict(X).tolist()[0]
    return {'prediction': int(pred)}
```
Run server:
```bash
uvicorn app:app --reload
```
Send a request (new terminal):
```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"features": [5.1, 3.5, 1.4, 0.2]}'
```


## Exercises
1) Add input validation and friendly error messages.
2) Return class name instead of numeric id.
3) Save the app requirements to a minimal requirements.txt for deployment.